# Standalone Evaluation Study: Pure NumPy Random Forest

This notebook serves as a complete standalone reproduction and evaluation of the manual Random Forest classification engine up to Phase 3, with a Phase 4 added to evaluate the trained RF model using the NumPy manual approach.

It strictly uses relative paths to load the cached Parquet dataset.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import warnings

# Suppress UserWarnings from scikit-learn
warnings.filterwarnings("ignore", category=UserWarning)

# Find checkpoint file in relative and home downloads paths
home_dir = os.path.expanduser("~")
candidates = [
    "checkpoint_balanced_full.parquet",
    "models/checkpoint_balanced_full.parquet",
    "../presentation/models/checkpoint_balanced_full.parquet",
    "../IDS_Dashboard_Submission-20260613T121815Z-3-001/IDS_Dashboard_Submission/models/checkpoint_balanced_full.parquet",
    "../../Downloads/models-20260613T064206Z-3-001/models/checkpoint_balanced_full.parquet",
    os.path.join(home_dir, "Downloads/models-20260613T064206Z-3-001/models/checkpoint_balanced_full.parquet"),
    os.path.join(home_dir, "Downloads/checkpoint_balanced_full.parquet")
]

CHECKPOINT_FILE = None
for c in candidates:
    if os.path.exists(c):
        CHECKPOINT_FILE = c
        break

if CHECKPOINT_FILE is None:
    CHECKPOINT_FILE = "../presentation/models/checkpoint_balanced_full.parquet"

print(f"Using checkpoint file: {CHECKPOINT_FILE}")


Using checkpoint file: ../presentation/models/checkpoint_balanced_full.parquet


: 

## Phase 1: Data Ingestion and Feature Extraction

In [18]:
if os.path.exists(CHECKPOINT_FILE):
    print("Loading balanced dataset from cached Parquet...")
    df = pd.read_parquet(CHECKPOINT_FILE)
else:
    raise FileNotFoundError(f"Parquet file not found. Ensure {CHECKPOINT_FILE} is available.")

print(f"Dataset preprocessing complete. Total balanced rows: {len(df)}")

FileNotFoundError: Parquet file not found. Ensure ../presentation/models/checkpoint_balanced_full.parquet is available.

## Phase 2: Stratified Splitting and Normalization

In [ ]:
np.random.seed(42)
benign_idx = df[df['Label'] == 0].index.values.copy()
attack_idx = df[df['Label'] == 1].index.values.copy()

np.random.shuffle(benign_idx)
np.random.shuffle(attack_idx)

b_split = int(len(benign_idx) * 0.7)
a_split = int(len(attack_idx) * 0.7)

train_idx = np.concatenate([benign_idx[:b_split], attack_idx[:a_split]])
test_idx = np.concatenate([benign_idx[b_split:], attack_idx[a_split:]])

np.random.shuffle(train_idx)
np.random.shuffle(test_idx)

# Get feature column names
feature_cols = df.columns.drop('Label').tolist()

# Compute train set statistics using the full training set (representing standard behavior)
print("Computing train set statistics for normalization scaling...")
X_train_mean = df.loc[train_idx, feature_cols].mean(axis=0).values.astype(np.float32)
X_train_std = df.loc[train_idx, feature_cols].std(axis=0).values.astype(np.float32)
X_train_std[X_train_std == 0] = 1e-6

# Extract full test set directly and cast to float32
print("Extracting test set features...")
X_test_val = df.loc[test_idx, feature_cols].values.astype(np.float32)
y_test_val = df.loc[test_idx, 'Label'].values.astype(np.int32)

del df

# Normalize test set using train set statistics
X_test_val = (X_test_val - X_train_mean) / X_train_std

print(f"Splitting and normalization complete. Test Set size: {X_test_val.shape}")

## Phase 3: Random Forest Classification Engine (Definitions Only)

In [ ]:
class Node:
    def __init__(self, feature_idx=None, threshold=None, left=None, right=None, *, value=None):
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
    def is_leaf(self): return self.value is not None

class DecisionTree:
    def __init__(self, max_depth=5, min_samples_split=5):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.root = self._grow_tree(X, y, 0)

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape
        if depth >= self.max_depth or len(np.unique(y)) == 1 or n_samples < self.min_samples_split:
            return Node(value=self._most_common_label(y))

        feat_idxs = np.random.choice(n_features, int(np.sqrt(n_features)), replace=False)
        best_feat, best_thresh = self._best_criteria(X, y, feat_idxs)

        if best_feat is None: return Node(value=self._most_common_label(y))

        left_idxs, right_idxs = self._split(X[:, best_feat], best_thresh)
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        return Node(best_feat, best_thresh, left, right)

    def _best_criteria(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_thresh = None, None
        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.percentile(X_column, [20, 40, 60, 80])
            for threshold in thresholds:
                gain = self._information_gain(y, X_column, threshold)
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = threshold
        return split_idx, split_thresh

    def _information_gain(self, y, X_column, split_thresh):
        parent_gini = self._gini(y)
        left_idxs, right_idxs = self._split(X_column, split_thresh)
        if len(left_idxs) == 0 or len(right_idxs) == 0: return 0
        n = len(y)
        child_gini = (len(left_idxs)/n)*self._gini(y[left_idxs]) + (len(right_idxs)/n)*self._gini(y[right_idxs])
        return parent_gini - child_gini

    def _split(self, X_column, split_thresh):
        return np.argwhere(X_column <= split_thresh).flatten(), np.argwhere(X_column > split_thresh).flatten()

    def _gini(self, y):
        proportions = np.bincount(y.astype(int)) / len(y)
        return 1 - np.sum([p**2 for p in proportions if p > 0])

    def _most_common_label(self, y):
        return 0 if len(y) == 0 else np.bincount(y.astype(int)).argmax()

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf(): return node.value
        if x[node.feature_idx] <= node.threshold: return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

class PureNumpyRandomForest:
    def __init__(self, n_trees=200, max_depth=20):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.trees = []

    def fit(self, X, y):
        for i in range(self.n_trees):
            tree = DecisionTree(max_depth=self.max_depth)
            idxs = np.random.choice(X.shape[0], X.shape[0], replace=True)
            tree.fit(X[idxs], y[idxs])
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = np.swapaxes(np.array([tree.predict(X) for tree in self.trees]), 0, 1)
        return np.array([np.bincount(pred).argmax() for pred in tree_preds])

print("Random Forest Engine definitions loaded. Proceed to loading existing PKL model.")

## Phase 4: Model Load & Performance Evaluation (NumPy Manual Approach)

In [ ]:
import joblib
# Define target output file paths to inspect
home_dir = os.path.expanduser("~")
pkl_candidates = [
    "rf_ids_cic.pkl",
    "models/rf_ids_cic.pkl",
    "../presentation/models/rf_ids_cic.pkl",
    "../IDS_Dashboard_Submission-20260613T121815Z-3-001/IDS_Dashboard_Submission/models/rf_ids_cic.pkl",
    "../../Downloads/models-20260613T064206Z-3-001/models/rf_ids_cic.pkl",
    os.path.join(home_dir, "Downloads/models-20260613T064206Z-3-001/models/rf_ids_cic.pkl"),
    os.path.join(home_dir, "Downloads/rf_ids_cic.pkl")
]

pkl_exists = False
existing_path = ""
for path in pkl_candidates:
    if os.path.exists(path):
        pkl_exists = True
        existing_path = path
        break

if not pkl_exists:
    raise FileNotFoundError("Pre-trained model checkpoint (rf_ids_cic.pkl) not found. Please ensure it is present in one of the expected paths.")

print(f"Loading pre-trained model checkpoint from: {existing_path}")
rf_model = joblib.load(existing_path)

def evaluate_metrics(y_true, y_pred, title):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true) if len(y_true) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    print("=" * 60)
    print(f"             {title}")
    print("=" * 60)
    print(f"  Accuracy: {accuracy*100:.2f}% | Recall: {recall*100:.2f}%")
    print(f"  TP: {tp:,} | FP: {fp:,} | FN: {fn:,} | TN: {tn:,}")
    print("=" * 60)

print("Evaluating model predictions on the full test set...")
# Convert test set to DataFrame with column names to match model expectations and prevent UserWarnings
X_test_df = pd.DataFrame(X_test_val, columns=feature_cols)
y_pred_val = rf_model.predict(X_test_df)
evaluate_metrics(y_test_val, y_pred_val, "PRE-TRAINED RF MODEL EVALUATION (FULL TEST SET)")
